In [1]:
# Data importation, manipulation, and storage
from src.functions import import_data
import json 
import numpy as np
import pandas as pd
from urllib.request import urlopen

# Plotting
import plotly.express as px

In [2]:
# Data loading

# The county GeoJSON provides polygon dimensions for each US county.
with urlopen("http://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json") as response: 
    counties = json.load(response)

dfU = import_data("data/unemployment2023.csv")
dfCL = import_data("data/cost_of_living_us.csv")

In [3]:
# Update "FIPS_Code" to match the format in the Counties GeoJSON
# The first few states have codes of 1001 rather than "01001". They need to match.
dfU["FIPS_Code"] = dfU["FIPS_Code"].astype(str)
dfU['FIPS_Code'] = dfU['FIPS_Code'].str.zfill(5)

In [4]:
# Cleaning dfU (Unemployment)

# Filter the dataframe to only show the "Median Household Income" attribute
dfU= dfU[dfU['Attribute'].str.contains("Median_Household_Income")]
del dfU["Attribute"]
dfU.rename(columns={"Value": "median_household_income"}, inplace=True)

# Rename columns to align with the style of dfCL
dfU.rename(columns={"Area_Name": "area_name", "FIPS_Code": "fips_code",
                    "State": "state"}, inplace=True)

dfU.head()

,fips_code,state,area_name,median_household_income
96,00000,US,United States,74755.0
193,01000,AL,Alabama,59703.0
294,01001,AL,"Autauga County, AL",70148.0
395,01003,AL,"Baldwin County, AL",71704.0
496,01005,AL,"Barbour County, AL",41151.0


In [5]:
# Clean dfCL (Cost of Living)

# Create a new column that matches the column in dfU
dfCL["area_name"] = dfCL["county"] + ", " + dfCL["state"]

# Delete redundant columns
del dfCL["areaname"]
del dfCL["state"]
del dfCL["county"]

# Rename columns to align with the existing style
dfCL.rename(columns={"isMetro": "is_metro"}, inplace=True)

dfCL.head()

,case_id,is_metro,family_member_count,housing_cost,food_cost,transportation_cost,healthcare_cost,other_necessities_cost,childcare_cost,taxes,total_cost,median_family_income,area_name
0,1,True,1p0c,8505.72876,3454.91712,10829.16876,5737.47984,4333.81344,0.0000,6392.94504,39254.0532,73010.414062,"Autauga County, AL"
1,1,True,1p1c,12067.50240,5091.70788,11588.19288,8659.55640,6217.45896,6147.8298,7422.07836,57194.3256,73010.414062,"Autauga County, AL"
2,1,True,1p2c,12067.50240,7460.20308,12361.77720,11581.63260,7075.65816,15824.6940,9769.56228,76141.0308,73010.414062,"Autauga County, AL"
3,1,True,1p3c,15257.15040,9952.23924,13452.18600,14503.70760,9134.35620,18802.1892,13101.70320,94203.5328,73010.414062,"Autauga County, AL"
4,1,True,1p4c,15257.15040,12182.21400,13744.59840,17425.78560,9942.36396,18802.1892,13469.21880,100823.5200,73010.414062,"Autauga County, AL"


In [6]:
# Finally, merge dfCl and dfU
df = pd.merge(dfU, dfCL, how='left', left_on='area_name', right_on='area_name')

df.head()

,fips_code,state,area_name,median_household_income,case_id,is_metro,family_member_count,housing_cost,food_cost,transportation_cost,healthcare_cost,other_necessities_cost,childcare_cost,taxes,total_cost,median_family_income
0,00000,US,United States,74755.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01000,AL,Alabama,59703.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01001,AL,"Autauga County, AL",70148.0,1.0,True,1p0c,8505.72876,3454.91712,10829.16876,5737.47984,4333.81344,0.0000,6392.94504,39254.0532,73010.414062
3,01001,AL,"Autauga County, AL",70148.0,1.0,True,1p1c,12067.50240,5091.70788,11588.19288,8659.55640,6217.45896,6147.8298,7422.07836,57194.3256,73010.414062
4,01001,AL,"Autauga County, AL",70148.0,1.0,True,1p2c,12067.50240,7460.20308,12361.77720,11581.63260,7075.65816,15824.6940,9769.56228,76141.0308,73010.414062


In [7]:
# Save the resulting dataframe to a CSV to be imported again later
df.to_csv("data/us_metrics.csv")